In [1]:
# ============================================================
# STEP 1: LOAD THE SAME DATASET AS METHODS 1-3
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path

CSV_PATH = Path('/content/neo_asteroid_dataset.csv')
if not CSV_PATH.is_file():
    # Uploaded dataset, available in this ChatGPT session.
    local_csv = Path('/mnt/data/neo_asteroid_dataset(1).csv')
    if local_csv.is_file():
        CSV_PATH = local_csv
    else:
        # Google Colab: upload the same CSV used for methods 1-3.
        from google.colab import files
        print('Upload neo_asteroid_dataset.csv')
        uploaded = files.upload()
        csv_files = [Path(p) for p in uploaded if p.lower().endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError('Please upload neo_asteroid_dataset.csv')
        CSV_PATH = next((p for p in csv_files if p.name == 'neo_asteroid_dataset.csv'), csv_files[0])

df = pd.read_csv(CSV_PATH)
print('Dataset shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nPHA counts:')
print(df['pha'].value_counts(dropna=False))
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False))
print('\nFirst five rows:')
print(df.head().to_string(index=False))


Dataset shape: (42815, 21)

Columns:
['spkid', 'pdes', 'full_name', 'kind', 'neo', 'pha', 'e', 'a', 'q', 'i', 'om', 'w', 'moid', 'H', 'diameter', 'albedo', 'first_obs', 'last_obs', 'data_arc', 'n_obs_used', 'condition_code']

PHA counts:
pha
N      40130
Y       2550
NaN      135
Name: count, dtype: int64

Missing values:
albedo            41611
diameter          41569
data_arc            437
pha                 135
moid                135
H                     9
condition_code        2
neo                   0
pdes                  0
spkid                 0
kind                  0
full_name             0
a                     0
w                     0
om                    0
i                     0
q                     0
e                     0
first_obs             0
last_obs              0
n_obs_used            0
dtype: int64

First five rows:
   spkid pdes                full_name kind neo pha      e     a     q     i     om      w   moid     H  diameter  albedo  first_obs   last_o

In [2]:
# ============================================================
# STEP 2: SAME FEATURES, TARGET AND PREPROCESSING AS METHODS 1-3
# ============================================================
FEATURES = ['a', 'e', 'i', 'om', 'w', 'diameter']
TARGET = 'pha'

missing_columns = [col for col in FEATURES + [TARGET] if col not in df.columns]
if missing_columns:
    raise ValueError(f'Missing columns: {missing_columns}')

data = df.dropna(subset=[TARGET]).copy()
data[TARGET] = data[TARGET].astype(str).str.strip().str.upper().map({'Y': 1, 'N': 0})
data = data.dropna(subset=[TARGET]).copy()
data[TARGET] = data[TARGET].astype(int)

for col in FEATURES:
    data[col] = pd.to_numeric(data[col], errors='coerce')

X = data[FEATURES]
y = data[TARGET]
print('Prepared dataset:', data.shape)
print('\nTarget distribution:')
print(y.value_counts())
print('\nMissing values in selected features BEFORE handling:')
print(X.isna().sum())
print('\nDiameter is included in the six model features:', 'diameter' in X.columns)


Prepared dataset: (42680, 21)

Target distribution:
pha
0    40130
1     2550
Name: count, dtype: int64

Missing values in selected features BEFORE handling:
a               0
e               0
i               0
om              0
w               0
diameter    41434
dtype: int64

Diameter is included in the six model features: True


In [3]:
# ============================================================
# STEP 3: SAME STRATIFIED 80/20 SPLIT AS METHODS 1-3
# ============================================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Missing diameter in training:', X_train['diameter'].isna().sum())
print('Missing diameter in testing:', X_test['diameter'].isna().sum())


Training rows: 34144
Testing rows: 8536
Missing diameter in training: 33145
Missing diameter in testing: 8289


In [4]:
# ============================================================
# STEP 4: COMPLETE-CASE ANALYSIS + SAME CLASSIFIER AS METHODS 1 AND 3
# ============================================================
from sklearn.ensemble import HistGradientBoostingClassifier

# Preserve the same original 80/20 split, then remove rows with ANY
# missing model feature in EACH partition. No imputation is performed.
train_complete_mask = X_train.notna().all(axis=1)
test_complete_mask = X_test.notna().all(axis=1)

X_train_prepared = X_train.loc[train_complete_mask].copy()
y_train_prepared = y_train.loc[train_complete_mask].copy()
X_test_prepared = X_test.loc[test_complete_mask].copy()
y_test_prepared = y_test.loc[test_complete_mask].copy()

print('Original training rows:', len(X_train))
print('Complete training rows:', len(X_train_prepared))
print('Original testing rows:', len(X_test))
print('Complete testing rows:', len(X_test_prepared))
print('\nComplete-case training label distribution:')
print(y_train_prepared.value_counts())
print('\nComplete-case test label distribution:')
print(y_test_prepared.value_counts())

if y_train_prepared.nunique() != 2 or y_test_prepared.nunique() != 2:
    raise ValueError('Both classes are needed in complete-case train and test sets.')

model = HistGradientBoostingClassifier(
    max_iter=100, learning_rate=0.1,
    random_state=42, class_weight='balanced'
)
model.fit(X_train_prepared, y_train_prepared)
print('\nMethod 6 model trained successfully!')
print('Missing-value imputation: None; incomplete rows excluded')
print('Missing train features:', int(X_train_prepared.isna().sum().sum()))
print('Missing test features:', int(X_test_prepared.isna().sum().sum()))
print('WARNING: Complete-case test set is smaller and has a different class balance.')


Original training rows: 34144
Complete training rows: 999
Original testing rows: 8536
Complete testing rows: 247

Complete-case training label distribution:
pha
0    733
1    266
Name: count, dtype: int64

Complete-case test label distribution:
pha
0    173
1     74
Name: count, dtype: int64



Method 6 model trained successfully!
Missing-value imputation: None; incomplete rows excluded
Missing train features: 0
Missing test features: 0


In [5]:
# ============================================================
# STEP 5: SAME EVALUATION METRICS AS METHODS 1-3
# ============================================================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

EVAL_METHOD = 'Complete-Case Analysis'
eval_y = y_test_prepared
y_pred = model.predict(X_test_prepared)
y_prob = model.predict_proba(X_test_prepared)[:, 1]

results = {
    'Method': EVAL_METHOD,
    'Training Rows': len(X_train_prepared),
    'Testing Rows': len(X_test_prepared),
    'Test PHA Prevalence': float(eval_y.mean()),
    'Accuracy': accuracy_score(eval_y, y_pred),
    'Precision': precision_score(eval_y, y_pred, zero_division=0),
    'Recall': recall_score(eval_y, y_pred, zero_division=0),
    'F1-score': f1_score(eval_y, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(eval_y, y_prob),
    'Average Precision': average_precision_score(eval_y, y_prob)
}

print('METHOD 6: COMPLETE-CASE ANALYSIS')
print('-' * 55)
for metric, value in results.items():
    if isinstance(value, (float, np.floating)):
        print(f'{metric}: {value:.4f}')
    else:
        print(f'{metric}: {value}')
print('\nConfusion Matrix:')
print(confusion_matrix(eval_y, y_pred))
print('\nClassification Report:')
print(classification_report(eval_y, y_pred, target_names=['Non-PHA', 'PHA'], zero_division=0))


METHOD 6: COMPLETE-CASE ANALYSIS
-------------------------------------------------------
Method: Complete-Case Analysis
Training Rows: 999
Testing Rows: 247
Test PHA Prevalence: 0.2996
Accuracy: 0.7611
Precision: 0.6271
Recall: 0.5000
F1-score: 0.5564
ROC-AUC: 0.7763
Average Precision: 0.6329

Confusion Matrix:
[[151  22]
 [ 37  37]]

Classification Report:
              precision    recall  f1-score   support

     Non-PHA       0.80      0.87      0.84       173
         PHA       0.63      0.50      0.56        74

    accuracy                           0.76       247
   macro avg       0.72      0.69      0.70       247
weighted avg       0.75      0.76      0.75       247



In [6]:
# ============================================================
# STEP 6: COMPARABILITY WARNING
# ============================================================
print('Method 6 uses the same complete-case test subset as the secondary')
print('comparison in Methods 4 and 5 (same train/test split seed).')
print('But unlike those methods, it trains on only complete cases.')
print('Do NOT directly compare its 247-row primary score against')
print('the 8,536-row primary scores for imputation methods.')


Method 6 uses the same complete-case test subset as the secondary
comparison in Methods 4 and 5 (same train/test split seed).
But unlike those methods, it trains on only complete cases.
Do NOT directly compare its 247-row primary score against
the 8,536-row primary scores for imputation methods.


In [7]:
# ============================================================
# STEP 7: EXPORT RESULTS (SAME METRIC NAMES)
# ============================================================
results_df = pd.DataFrame([results])
results_df.to_csv('06_complete_case_analysis_results.csv', index=False)
print('Saved: 06_complete_case_analysis_results.csv')
print(results_df.to_string(index=False))


Saved: 06_complete_case_analysis_results.csv
                Method  Training Rows  Testing Rows  Test PHA Prevalence  Accuracy  Precision  Recall  F1-score  ROC-AUC  Average Precision
Complete-Case Analysis            999           247             0.299595  0.761134   0.627119     0.5  0.556391 0.776285           0.632905
